# Phase 10: Storage Tank Monitoring & Biochemical Deterioration Analysis
### AI-Driven Intelligent Greywater Management and Smart Reuse Routing System

---
### Scientific Disclaimer and Scope Notice
> **"The current project dataset is cross-sectional and does NOT contain longitudinal storage experiments. Therefore, this module provides a configurable mechanistic/operational deterioration estimate rather than an empirically validated prediction of exact greywater shelf life."**
>
> Data and estimation categories are explicitly tagged throughout this notebook:
> - `[OBSERVED DATA]`: Empirical distribution statistics from the 1,500 physical water samples in `dataset/main.csv`.
> - `[SYNTHETIC DATA]`: Discrete-time telemetry produced by the Phase 8 Digital Twin simulation.
> - `[MODEL ESTIMATE]`: Computed first-order kinetic and multi-factor index values.
> - `[UNVALIDATED ASSUMPTION]`: Kinetic rate constants ($k_{ref}$), activation energies ($E_a$), and temperature coefficients ($	heta$).



## 1. Objective

The objective of Phase 10 is to build a physical storage tank state tracking and biochemical degradation estimation engine.

When greywater is diverted to storage (e.g. when Phase 9 weather suppresses landscape irrigation due to rainfall), water is retained in tanks where organic matter continues to exert biochemical oxygen demand (BOD/COD), dissolved oxygen (DO) depletes toward anoxia, and stagnation risks accumulate.

### End-to-End System Context
```
Digital Twin / Sensors  [SYNTHETIC DATA]
          ↓
Water Quality Screening  [OBSERVED DATA]
          ↓
Isolation Forest (Safety Filter)
          ↓
XGBoost Initial Route
          ↓
Weather Context (Phase 9)
          ↓
[STORAGE & BIOCHEMICAL DECAY MONITORING] <--- PHASE 10
          ↓
Final Smart Routing (Phase 11)
```



In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import storage
from storage.config import (
    DEFAULT_TANK_CAPACITY_L,
    DEFAULT_INITIAL_LEVEL_L,
    DECAY_PARAMETERS,
    DETERIORATION_BANDS,
    DETERIORATION_WEIGHTS,
    STAGNATION_THRESHOLDS
)
from storage.storage_manager import StorageTank
from storage.decay_model import BiochemicalDecayModel
from storage.shelf_life import StorageMonitor
from simulation.storage_adapter import StorageSimulationAdapter, run_storage_simulation

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10

print("Storage Subsystem and Configuration loaded successfully.")
print(f"Default Tank Capacity: {DEFAULT_TANK_CAPACITY_L} L")



## 2. Digital Twin Telemetry Ingestion `[SYNTHETIC DATA]`

We load the 24-hour synthetic telemetry generated in Phase 8 (`simulation/data/synthetic_telemetry.csv`), spanning 288 timesteps at 5-minute intervals.



In [ ]:
telemetry_path = os.path.join(project_root, "simulation", "data", "synthetic_telemetry.csv")
df_telemetry = pd.read_csv(telemetry_path)
print(f"Loaded Digital Twin telemetry: {df_telemetry.shape[0]} rows, {df_telemetry.shape[1]} columns")
df_telemetry[['timestamp', 'greywater_source', 'flow_rate_L_min', 'TEMP_C', 'BOD_mg_L', 'COD_mg_L', 'DO_mg_L', 'event_name']].head()



## 3. Storage Tank Initialization & Physical Mass Balance `[MODEL ESTIMATE]`

The `StorageTank` enforces physical conservation of volume:
$$V_t = V_{t-1} + V_{in} - V_{out}$$
Subject to non-negativity and capacity constraints:
$$0 \le V_t \le V_{capacity}$$
Attempts to overflow or withdraw more than available water raise explicit exceptions.



In [ ]:
# Initialize tank with 250 L
tank = StorageTank(
    tank_id="TANK_DEMO_01",
    capacity_liters=1000.0,
    initial_level_liters=250.0,
    initial_timestamp="2026-10-02 06:00:00 UTC",
    initial_temperature_C=22.0
)

state = tank.get_state()
print(f"Tank ID:             {state['tank_id']}")
print(f"Current Level:       {state['current_level_liters']} L ({state['fill_percentage']}%)")
print(f"Available Capacity:  {state['available_capacity_liters']} L")
print(f"Initial Batches:     {state['batch_count']}")



## 4. Tank Inflow and Outflow Simulation `[MODEL ESTIMATE]`

We run the full 24-hour simulation across all 288 steps using `StorageSimulationAdapter`.
Inflows vary based on greywater source flow rates, while intermittent outflows represent treatment pump discharges.



In [ ]:
# Run complete storage simulation
output_csv = os.path.join(project_root, "simulation", "data", "storage_monitoring_output.csv")
df_sim = run_storage_simulation(telemetry_path=telemetry_path, output_path=output_csv)
print(f"Simulation completed with {len(df_sim)} records.")
df_sim[['timestamp', 'source', 'current_level_liters', 'fill_percentage', 'storage_age_hours', 'deterioration_index', 'recommended_storage_action']].head(10)



In [ ]:
# Plot: Tank Level vs Time
plt.figure(figsize=(12, 4))
plt.plot(df_sim.index, df_sim['current_level_liters'], color='#1f77b4', linewidth=2, label='Current Water Level (L)')
plt.axhline(1000.0, color='#d62728', linestyle='--', linewidth=1.5, label='Maximum Capacity (1000 L)')
plt.axhline(0.0, color='gray', linestyle=':', linewidth=1)
plt.title('Figure 1: Tank Water Level Progression Over 24 Hours [MODEL ESTIMATE]')
plt.xlabel('Simulation Timestep (5-minute intervals)')
plt.ylabel('Fluid Volume (Liters)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()



## 5. Storage Age Analysis & Batch Tracking `[MODEL ESTIMATE]`

Rather than blindly resetting the tank age on every inflow, the `StorageTank` implements volume-weighted FIFO batch tracking:
$$\text{Age}_{weighted} = \frac{\sum_{i} (V_i \times \text{Age}_i)}{\sum_i V_i}$$
This correctly models fluid retention without losing track of older fluid volumes.



In [ ]:
# Plot: Storage Age vs Time
plt.figure(figsize=(12, 4))
plt.plot(df_sim.index, df_sim['storage_age_hours'], color='#2ca02c', linewidth=2, label='Volume-Weighted Storage Age (hours)')
plt.title('Figure 2: Storage Age Progression Over 24 Hours [MODEL ESTIMATE]')
plt.xlabel('Simulation Timestep (5-minute intervals)')
plt.ylabel('Storage Age (Hours)')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()



## 6. Temperature Context `[SYNTHETIC DATA]` & `[UNVALIDATED ASSUMPTION]`

Temperature is acquired from sensor telemetry or Phase 9 weather context.
Higher temperatures increase biological degradation kinetics according to the Arrhenius relationship:
$$k(T) = k_{ref} \cdot \exp\left[-\frac{E_a}{R}\left(\frac{1}{T} - \frac{1}{T_{ref}}\right)\right]$$
*Parameter status:* `UNVALIDATED`.



In [ ]:
decay_model = BiochemicalDecayModel()
temps = np.linspace(10.0, 40.0, 31)
rates_bod = [decay_model.calculate_adjusted_rate("BOD_mg_L", t) for t in temps]
rates_cod = [decay_model.calculate_adjusted_rate("COD_mg_L", t) for t in temps]

plt.figure(figsize=(10, 4))
plt.plot(temps, rates_bod, color='#e377c2', linewidth=2, label='BOD Decay Rate k(T) [hr⁻¹]')
plt.plot(temps, rates_cod, color='#7f7f7f', linewidth=2, linestyle='--', label='COD Decay Rate k(T) [hr⁻¹]')
plt.axvline(20.0, color='blue', linestyle=':', label='Reference Temp (20°C)')
plt.title('Figure 3: Temperature-Adjusted Decay Rate k(T) (Arrhenius Kinetics) [UNVALIDATED ASSUMPTION]')
plt.xlabel('Temperature (°C)')
plt.ylabel('Decay Rate Constant (hr⁻¹)')
plt.legend()
plt.tight_layout()
plt.show()



## 7. Biochemical Decay Model `[UNVALIDATED ASSUMPTION]`

We compare parameters across their model categories:
- **`SUPPORTED_KINETIC_MODEL`**: BOD, COD (first-order decay).
- **`RELATIVE_RISK_PROXY`**: DO (depletion), Turbidity (settling).
- **`NOT_MODELED`**: *E. coli* (`microbial_risk_status = "NOT_EMPIRICALLY_VALIDATED"`).



In [ ]:
hours_range = np.linspace(0, 48, 49)
bod_curve = [decay_model.estimate_parameter_decay("BOD_mg_L", 200.0, h, 20.0)["estimated_value"] for h in hours_range]
cod_curve = [decay_model.estimate_parameter_decay("COD_mg_L", 400.0, h, 20.0)["estimated_value"] for h in hours_range]
do_curve = [decay_model.estimate_parameter_decay("DO_mg_L", 6.0, h, 20.0)["estimated_value"] for h in hours_range]

fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.plot(hours_range, bod_curve, color='#ff7f0e', label='BOD (mg/L)')
ax1.plot(hours_range, cod_curve, color='#9467bd', label='COD (mg/L)')
ax1.set_xlabel('Storage Duration (Hours)')
ax1.set_ylabel('BOD / COD (mg/L)', color='black')

ax2 = ax1.twinx()
ax2.plot(hours_range, do_curve, color='#17becf', linestyle='--', linewidth=2, label='Dissolved Oxygen (mg/L)')
ax2.set_ylabel('DO (mg/L)', color='#17becf')

plt.title('Figure 4: Modeled Parameter Decay Over 48 Hours at 20°C [UNVALIDATED ASSUMPTION]')
fig.tight_layout()
plt.show()



## 8. Multi-Factor Deterioration Index `[MODEL ESTIMATE]`

The composite deterioration index integrates:
1. Operational storage age factor (35%)
2. Temperature acceleration factor (20%)
3. Organic matter decay progress (20%)
4. Dissolved oxygen depletion progress (15%)
5. Stagnation dormancy factor (10%)



In [ ]:
# Plot: Deterioration Index vs Time
plt.figure(figsize=(12, 4))
plt.plot(df_sim.index, df_sim['deterioration_index'], color='#d62728', linewidth=2, label='Composite Deterioration Index')
plt.axhline(0.25, color='gold', linestyle='--', label='Moderate Deterioration (0.25)')
plt.axhline(0.50, color='orange', linestyle='--', label='High Deterioration (0.50)')
plt.axhline(0.75, color='red', linestyle='--', label='Critical Review (0.75)')
plt.title('Figure 5: Composite Deterioration Index Progression Over 24 Hours [MODEL ESTIMATE]')
plt.xlabel('Simulation Timestep (5-minute intervals)')
plt.ylabel('Deterioration Index [0.0 - 1.0]')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()



## 9. Stagnation Detection `[MODEL ESTIMATE]`

Operational stagnation checks identify idle water volumes where low turnover risks anaerobic septic formation:
- `NORMAL`: Regular turnover, idle $< 12$ hours.
- `LOW_TURNOVER`: Idle $12$ to $24$ hours.
- `STAGNANT`: Idle $\ge 24$ hours.
- `HIGH_STAGNATION_RISK`: Idle $\ge 36$ hours or idle $\ge 24$ hours with deterioration index $\ge 0.50$.



In [ ]:
stagnation_counts = df_sim['stagnation_status'].value_counts()
print("Stagnation Status Distribution across 24h simulation:")
for status, count in stagnation_counts.items():
    pct = (count / len(df_sim)) * 100
    print(f"  {status:25s}: {count:3d} steps ({pct:.1f}%)")



## 10. Operational Shelf-Life Categorization

Due to lack of empirical longitudinal data, exact safe storage hours are not claimed:
$$\text{estimated\_remaining\_time\_hours} = \text{null}$$
Instead, qualitative operational bands guide facility management:
`FRESH`, `MONITOR`, `AGING`, `HIGH_RISK_REVIEW`.



In [ ]:
shelf_counts = df_sim['shelf_life_status'].value_counts()
action_counts = df_sim['recommended_storage_action'].value_counts()

print("Shelf-Life Status Distribution:")
for k, v in shelf_counts.items():
    print(f"  {k:25s}: {v:3d} steps ({(v/len(df_sim))*100:.1f}%)")

print()
print("Recommended Storage Actions:")
for k, v in action_counts.items():
    print(f"  {k:25s}: {v:3d} steps ({(v/len(df_sim))*100:.1f}%)")



In [ ]:
# Plot: Source-Wise Storage Behavior and Inflow Volumes
plt.figure(figsize=(10, 4))
source_grouped = df_sim.groupby('source')['inflow_volume_liters'].sum()
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
bars = plt.bar(source_grouped.index, source_grouped.values, color=colors, edgecolor='black', alpha=0.85)
plt.title('Figure 6: Cumulative Inflow Volume by Greywater Source [MODEL ESTIMATE]')
plt.xlabel('Greywater Source Category')
plt.ylabel('Total Inflow Volume (Liters)')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 10, f"{yval:.1f} L", ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()



## 11. Controlled Scenario Verification

We verify the 6 required testing scenarios:
- **Scenario A**: Fresh inflow $\rightarrow$ low storage age ($< 1$ hr).
- **Scenario B**: Continuous inflow $\rightarrow$ increasing tank level.
- **Scenario C**: Long storage period $\rightarrow$ increasing deterioration index ($> 0.50$).
- **Scenario D**: Higher temperature $\rightarrow$ accelerated kinetic degradation.
- **Scenario E**: No outflow for extended period $\rightarrow$ stagnation warning (`STAGNANT`).
- **Scenario F**: High-risk water quality input $\rightarrow$ immediate `SAFETY_REVIEW`.



In [ ]:
monitor = StorageMonitor()

# Scenario A: Fresh Inflow
tank_a = StorageTank(capacity_liters=1000.0, initial_level_liters=0.0)
tank_a.add_inflow(150.0, "2026-10-02 08:00:00 UTC")
state_a = tank_a.get_state("2026-10-02 08:30:00 UTC")
eval_a = monitor.assess_shelf_life(state_a)
print("Scenario A (Fresh Inflow):")
print(f"  Storage Age: {state_a['storage_age_hours']:.1f}h | Index: {eval_a['deterioration_index']} | Status: {eval_a['shelf_life_status']}")
print()

# Scenario C: Long Storage Period
tank_c = StorageTank(capacity_liters=1000.0, initial_level_liters=300.0, initial_timestamp="2026-10-01 00:00:00 UTC")
state_c = tank_c.get_state("2026-10-02 12:00:00 UTC")  # 36 hours later
eval_c = monitor.assess_shelf_life(state_c)
print("Scenario C (Long Storage 36h):")
print(f"  Storage Age: {state_c['storage_age_hours']:.1f}h | Index: {eval_c['deterioration_index']} | Status: {eval_c['shelf_life_status']} | Action: {eval_c['recommended_storage_action']}")
print()

# Scenario D: High Temperature Comparison
eval_d_cool = monitor.assess_shelf_life({**state_c, "storage_temperature_C": 18.0})
eval_d_warm = monitor.assess_shelf_life({**state_c, "storage_temperature_C": 35.0})
print("Scenario D (Temperature Sensitivity at 36h):")
print(f"  Cool (18°C) Index: {eval_d_cool['deterioration_index']} vs Warm (35°C) Index: {eval_d_warm['deterioration_index']}")
print()

# Scenario E: Stagnation (30h idle)
eval_e = monitor.assess_shelf_life({**state_c, "last_outflow_timestamp": None})
print("Scenario E (Extended Inactivity):")
print(f"  Stagnation Status: {eval_e['stagnation_status']} | Action: {eval_e['recommended_storage_action']}")
print()

# Scenario F: High-Risk Upstream Input
eval_f = monitor.assess_shelf_life(state_a, initial_water_quality_status="SEWER_BYPASS_OVERRIDE")
print("Scenario F (High-Risk Anomaly Safety Override):")
print(f"  Recommendation: {eval_f['recommended_storage_action']} (Safety precedence enforced)")



## 12. Scientific Limitations & Assumptions

1. **Absence of Longitudinal Decay Measurements**: The cross-sectional dataset contains no time-series measurements of stored greywater samples. Kinetic models are mechanistic approximations for simulation.
2. **Microbial Pathogen Unpredictability**: Bacteria such as *E. coli* do not follow monotonic first-order decay without specific disinfection or nutrient exhaustion assays. Microbial risk is designated `NOT_EMPIRICALLY_VALIDATED`.
3. **Tank Mixing Dynamics**: Fluid mixing is modeled as discrete batches rather than complete 3D computational fluid dynamics (CFD).
4. **No Health Standards Claim**: Deterioration indices are operational management indices and not regulatory health guarantees.



## 13. Conclusions & Phase 11 Readiness

- Physical storage tank accounting guarantees mass balance ($0 \le V \le 1000\text{ L}$).
- FIFO batch tracking preserves fluid age accurately without artificial age resets.
- Multi-factor deterioration indices and stagnation detection provide operational decision support.
- Upstream safety overrides strictly supersede storage retention.
- All Phase 1–10 tests pass (79/79).
- **The system is fully prepared for Phase 11 (Smart Context-Aware Routing Engine).**

